# PL ?????? (PYNQ)

??: ???? AXI-Lite + AXI DMA ??????????????

????: **???? B????? A**??? input bank swap ?????


In [ ]:
import time
import numpy as np
import pynq

# AXI-Lite register map (axi_lite_control.v)
REG_CTRL      = 0x00
REG_STATUS    = 0x04
REG_CFG_SEQ   = 0x08
REG_CFG_ACC   = 0x0C
REG_VERSION   = 0x10
REG_PPU_MULT  = 0x14
REG_PPU_SHIFT = 0x18
REG_PPU_ZP    = 0x1C
REG_PPU_BIAS  = 0x20
REG_OUT_EN    = 0x24

def calc_pad(x, align):
    return ((x + align - 1) // align) * align

def dma_status(ch):
    return ch._mmio.read(0x04)

def wait_dma_idle(ch, timeout_s=2.0):
    t0 = time.time()
    while time.time() - t0 < timeout_s:
        sr = dma_status(ch)
        if sr & 0x2:
            return True, sr
        time.sleep(0.001)
    return False, dma_status(ch)

def dump_regs(axi_ctrl):
    regs = {
        'CTRL': axi_ctrl.read(REG_CTRL),
        'STATUS': axi_ctrl.read(REG_STATUS),
        'CFG_SEQ': axi_ctrl.read(REG_CFG_SEQ),
        'CFG_ACC': axi_ctrl.read(REG_CFG_ACC),
        'VERSION': axi_ctrl.read(REG_VERSION),
        'PPU_MULT': axi_ctrl.read(REG_PPU_MULT),
        'PPU_SHIFT': axi_ctrl.read(REG_PPU_SHIFT),
        'PPU_ZP': axi_ctrl.read(REG_PPU_ZP),
        'PPU_BIAS': axi_ctrl.read(REG_PPU_BIAS),
        'OUT_EN': axi_ctrl.read(REG_OUT_EN),
    }
    print('=== AXI-Lite Register Dump ===')
    for k, v in regs.items():
        print(f'{k:>8} = 0x{v:08x}')
    return regs

def soft_reset(axi_ctrl):
    axi_ctrl.write(REG_CTRL, 0x00)
    time.sleep(0.01)
    axi_ctrl.write(REG_CTRL, 0x02)
    time.sleep(0.01)

def start_pulse(axi_ctrl):
    axi_ctrl.write(REG_CTRL, 0x03)
    axi_ctrl.write(REG_CTRL, 0x02)


In [ ]:
# ===== User Config =====
BIT_PATH = './deit/deit_accel.bit'
IP_NAME  = 'deit_accelerator_top_0'
DMA_NAME = 'axi_dma_0'

M, K, N = 2, 12, 16
TIMEOUT_S = 2.0
INPUT_DELAY_MS = 5
# =======================

overlay = pynq.Overlay(BIT_PATH)
axi_ctrl = getattr(overlay, IP_NAME)
dma = getattr(overlay, DMA_NAME)

m_pad = M if M % 2 == 0 else M + 1
k_pad = calc_pad(K, ARRAY_ROW)
n_pad = calc_pad(N, ARRAY_COL)

print(f'[INFO] M={M} K={K} N={N}')
print(f'[INFO] M_PAD={m_pad} K_PAD={k_pad} N_PAD={n_pad}')

# Build padded matrices
A = np.ones((m_pad, k_pad), dtype=np.int8)
B = np.ones((k_pad, n_pad), dtype=np.int8) * 2

# Pack streams to 64-bit beats
in_words = pack_input_stream(A)
w_words  = pack_weight_stream(B)

print(f'[INFO] input beats = {len(in_words)}')
print(f'[INFO] weight beats = {len(w_words)}')

# DMA buffers
buf_A = pynq.allocate(shape=(len(in_words),), dtype=np.uint64)
buf_B = pynq.allocate(shape=(len(w_words),), dtype=np.uint64)
buf_C = pynq.allocate(shape=(m_pad * n_pad,), dtype=np.int8)

buf_A[:] = in_words
buf_B[:] = w_words
buf_A.flush()
buf_B.flush()

# Config
soft_reset(axi_ctrl)
axi_ctrl.write(REG_CFG_SEQ, m_pad - 1)
axi_ctrl.write(REG_CFG_ACC, 0)
axi_ctrl.write(REG_PPU_MULT, 1)
axi_ctrl.write(REG_PPU_SHIFT, 0)
axi_ctrl.write(REG_PPU_ZP, 0)
axi_ctrl.write(REG_PPU_BIAS, 0)
axi_ctrl.write(REG_OUT_EN, 1)
dump_regs(axi_ctrl)

print('=== DMA Status (Before) ===')
print(f'MM2S SR = 0x{dma_status(dma.sendchannel):08x}')
print(f'S2MM SR = 0x{dma_status(dma.recvchannel):08x}')

# RX first
dma.recvchannel.transfer(buf_C)

# Start core
start_pulse(axi_ctrl)

# Send B then A
dma.sendchannel.transfer(buf_B)
ok_mm2s, sr_mm2s = wait_dma_idle(dma.sendchannel, TIMEOUT_S)
print(f'[INFO] MM2S-B idle={ok_mm2s}, SR=0x{sr_mm2s:08x}')

if INPUT_DELAY_MS > 0:
    time.sleep(INPUT_DELAY_MS / 1000.0)

dma.sendchannel.transfer(buf_A)
ok_mm2s_2, sr_mm2s_2 = wait_dma_idle(dma.sendchannel, TIMEOUT_S)
print(f'[INFO] MM2S-A idle={ok_mm2s_2}, SR=0x{sr_mm2s_2:08x}')

# AP_DONE polling
t0 = time.time()
done = False
while time.time() - t0 < TIMEOUT_S:
    if (axi_ctrl.read(REG_STATUS) & 0x1) != 0:
        done = True
        axi_ctrl.write(REG_STATUS, 0x1)
        break
    time.sleep(0.001)
print(f'[INFO] AP_DONE={done}, STATUS=0x{axi_ctrl.read(REG_STATUS):08x}')

# RX done
ok_s2mm, sr_s2mm = wait_dma_idle(dma.recvchannel, TIMEOUT_S)
print(f'[INFO] S2MM idle={ok_s2mm}, SR=0x{sr_s2mm:08x}')

buf_C.invalidate()
C_hw = np.array(buf_C).reshape((m_pad, n_pad))

C_golden = (A.astype(np.int32) @ B.astype(np.int32))
C_golden = np.clip(C_golden, -128, 127).astype(np.int8)

print('=== Validation (Row0) ===')
print('Golden:', C_golden[0])
print('HW    :', C_hw[0])

if ok_s2mm and np.array_equal(C_hw, C_golden):
    print('[PASS] DMA+TLAST+numeric OK')
elif np.array_equal(C_hw, C_golden):
    print('[WARN] Numeric OK but DMA not done (TLAST/length?)')
else:
    print('[FAIL] Numeric mismatch or stream issue')

buf_A.close(); buf_B.close(); buf_C.close()
